# Lab 10 - Notebook 02: Guided Wine Clustering Activity

**Student activity**

In this notebook you will cluster a real chemical-analysis dataset of wines. The dataset contains numerical measurements such as alcohol, magnesium, flavanoids, colour intensity and proline.

Although the original dataset also has known classes, **do not use the target labels for clustering**. Treat the feature table as unlabelled data.

Complete the notebook in order.

## Task 1 - Import the required tools

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score

RANDOM_STATE = 42

## Task 2 - Load and inspect the unlabelled feature data

Load the Wine dataset as a DataFrame. Use only the feature columns stored in `wine.data`.

In [2]:
wine = load_wine(as_frame=True)
X = wine.data.copy()

print('Shape:', X.shape)
display(X.head())
display(X.describe().T[['mean', 'std', 'min', 'max']].round(2))

Shape: (178, 13)


,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline
0,14.23,1.71,2.43,15.6,127.0,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065.0
1,13.20,1.78,2.14,11.2,100.0,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050.0
2,13.16,2.36,2.67,18.6,101.0,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185.0
3,14.37,1.95,2.50,16.8,113.0,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480.0
4,13.24,2.59,2.87,21.0,118.0,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735.0


,mean,std,min,max
alcohol,13.00,0.81,11.03,14.83
malic_acid,2.34,1.12,0.74,5.80
ash,2.37,0.27,1.36,3.23
alcalinity_of_ash,19.49,3.34,10.60,30.00
magnesium,99.74,14.28,70.00,162.00
total_phenols,2.30,0.63,0.98,3.88
flavanoids,2.03,1.00,0.34,5.08
nonflavanoid_phenols,0.36,0.12,0.13,0.66
proanthocyanins,1.59,0.57,0.41,3.58
color_intensity,5.06,2.32,1.28,13.00


### Practical preprocessing note

K-Means and DBSCAN are distance-based. Features measured on much larger numerical scales can dominate a distance calculation, so we standardize the numerical variables before clustering.

In [3]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print('Scaled matrix shape:', X_scaled.shape)
print('Approximate mean of first 3 scaled features:', np.round(X_scaled[:, :3].mean(axis=0), 3))
print('Approximate std of first 3 scaled features:', np.round(X_scaled[:, :3].std(axis=0), 3))

Scaled matrix shape: (178, 13)
Approximate mean of first 3 scaled features: [-0. -0. -0.]
Approximate std of first 3 scaled features: [1. 1. 1.]


## Task 3 - Use the Elbow Method

Run K-Means for **K = 1 to 8**, store the inertia for each K, and plot the result.

The code below is deliberately structured so you only need to complete the marked line.

In [4]:
inertias = []
k_values = range(1, 9)

for k in k_values:
    # TODO: create and fit a KMeans model using n_clusters=k, n_init=10 and RANDOM_STATE
    model = ______________________________
    model.fit(X_scaled)
    inertias.append(model.inertia_)

plt.figure(figsize=(6, 4))
plt.plot(list(k_values), inertias, marker='o')
plt.xlabel('Number of clusters (K)')
plt.ylabel('Inertia')
plt.title('Elbow Method - Wine data')
plt.show()

NameError: name '______________________________' is not defined

## Task 4 - Calculate Silhouette Coefficients

Calculate the silhouette coefficient for **K = 2 to 8**. A larger silhouette value generally indicates more compact and better-separated clusters.

In [ ]:
silhouette_results = []

for k in range(2, 9):
    # TODO: fit K-Means and obtain cluster labels
    model = ______________________________
    labels = ______________________________

    # TODO: calculate the silhouette coefficient
    score = ______________________________
    silhouette_results.append((k, score))

sil_df = pd.DataFrame(silhouette_results, columns=['K', 'Silhouette'])
display(sil_df.round(3))

plt.figure(figsize=(6, 4))
plt.plot(sil_df['K'], sil_df['Silhouette'], marker='o')
plt.xlabel('Number of clusters (K)')
plt.ylabel('Silhouette coefficient')
plt.title('Silhouette comparison - Wine data')
plt.show()

## Task 5 - Fit the final K-Means model

Use the elbow and silhouette evidence to choose a value for K. For this dataset, you should find that **K = 3** is a defensible choice.

Set `chosen_k` and fit the final model.

In [ ]:
chosen_k = 3

# TODO: fit the final K-Means model and obtain cluster labels
final_kmeans = ______________________________
kmeans_labels = ______________________________

print('Cluster sizes:')
print(pd.Series(kmeans_labels).value_counts().sort_index())
print('Final inertia:', round(final_kmeans.inertia_, 2))
print('Final silhouette:', round(silhouette_score(X_scaled, kmeans_labels), 3))

## Task 6 - Inspect the clusters

The clustering uses all standardized features, but we can display two original features to obtain a simple 2-D view.

In [ ]:
plt.figure(figsize=(6, 4))
plt.scatter(X['alcohol'], X['color_intensity'], c=kmeans_labels, s=35)
plt.xlabel('Alcohol')
plt.ylabel('Colour intensity')
plt.title('K-Means clusters - 2-D view')
plt.show()

# Convert learned centroids back to the original measurement scales.
centers_original = scaler.inverse_transform(final_kmeans.cluster_centers_)
centers_df = pd.DataFrame(centers_original, columns=X.columns)

display(centers_df[['alcohol', 'magnesium', 'flavanoids', 'color_intensity', 'proline']].round(2))

## Task 7 - Apply DBSCAN to the same scaled data

Use the supplied parameters first:

- `eps = 2.2`
- `min_samples = 5`

Then report how many clusters and noise points DBSCAN discovers.

In [ ]:
# TODO: create DBSCAN with eps=2.2 and min_samples=5
DBSCAN_model = ______________________________
dbscan_labels = ______________________________

n_clusters = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
n_noise = np.sum(dbscan_labels == -1)

print('DBSCAN clusters found:', n_clusters)
print('Noise points:', n_noise)
print('Labels present:', np.unique(dbscan_labels))

plt.figure(figsize=(6, 4))
plt.scatter(X['alcohol'], X['color_intensity'], c=dbscan_labels, s=35)
plt.xlabel('Alcohol')
plt.ylabel('Colour intensity')
plt.title('DBSCAN result - 2-D view')
plt.show()

## Task 8 - Short parameter check

Change only `eps` and observe the result:

- run once with `eps = 2.0`;
- run once with `eps = 2.5`.

Record the number of clusters and noise points. This demonstrates the parameter sensitivity discussed in the lecture.

In [ ]:
results = []

for eps_value in [2.0, 2.2, 2.5]:
    model = DBSCAN(eps=eps_value, min_samples=5)
    labels = model.fit_predict(X_scaled)
    clusters = len(set(labels)) - (1 if -1 in labels else 0)
    noise = np.sum(labels == -1)
    results.append([eps_value, clusters, noise])

pd.DataFrame(results, columns=['eps', 'Clusters', 'Noise points'])

## Questions to Answer

1. What value of **K** is supported by the elbow and silhouette results? Briefly justify your answer.
2. Why was standardization applied before using K-Means and DBSCAN on the Wine features?
3. What happened to the number of clusters and noise points when `eps` changed in DBSCAN? What does this tell you about the algorithm?

Keep each answer short and refer to the outputs generated in your notebook.